### Structured Output

In [2]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool

load_dotenv(override=True)

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [4]:
groq_model_name = os.getenv("GROQ_MODEL")
groq_model = init_chat_model(
    os.getenv("GROQ_MODEL"),
    model_provider="groq"
)
unstructured_response = groq_model.invoke("Give me details about the movie 'Jersey' acted by Nani.")
print(unstructured_response.content)

**Jersey (2019)**  
*Directed by: Gowri Krishna*  
*Produced by: S. S. Kanchi*  
*Starring: Nani (Srinivasa Rao), Nithya Menen, Prakash Raj, Prakash Raj (cameo), S. S. Kanchi, and others.*

---

### 1. Overview
- **Genre:** Sports drama / Family drama  
- **Language:** Telugu (original), Hindi dubbed version titled *Jersey* (2019) and Tamil dubbed *Jersey* (2020)  
- **Runtime:** 144 minutes  
- **Release Date:** 29 September 2019 (India)

### 2. Plot Summary
The story follows **Srinivasa Rao (Nani)**, a middle‑aged man who works as a bus conductor in a small town. He has a deep love for cricket and dreams of playing for the Indian national team, but life’s circumstances—financial struggles, a supportive wife (Nithya Menen), and a son (Prakash Raj) who later becomes a professional cricketer—force him to set aside his ambition.

Years later, when his son becomes an international cricketer, Srinivasa receives a golden opportunity: a chance to play in the Indian national team’s final matc

### Pydantic

- This model provides the richest feature set with`Field` validation, descriptions and nested structures.

### Create Schema using Pydantic's `BaseModel`

In [5]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    release_year: int = Field(..., description="The year the movie was released")
    genre: str = Field(..., description="The genre of the movie")

In [9]:
structured_response = groq_model.with_structured_output(Movie).invoke("Give me details about the movie 'Jersey' acted by Nani.")
structured_response

Movie(title='Jersey', release_year=2019, genre='Sports drama')

- You can see the difference between the unstructured and structured output. 
- The structured output is more predictable and easier to work with, as it adheres to the defined schema.
- With structured output, we can get only the required fields and avoid any unnecessary information.
- When we define `include_raw=True`, we can get both `AIMessage` and the `Structured Output`.

In [10]:
structured_response_with_ai_message = groq_model.with_structured_output(Movie, include_raw=True).invoke("Give me details about the movie 'Jersey' acted by Nani.")
structured_response_with_ai_message

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "Give me details about the movie \'Jersey\' acted by Nani." They want details about the movie. We need to call the function to get movie details. The function expects genre, release_year, title. We can provide these: genre: "Drama" or "Sports drama"? The movie "Jersey" starring Nani was released in 2019. Genre: Sports drama, romantic drama. We can call the function.', 'tool_calls': [{'id': 'fc_8a2f333f-a30a-45b7-9acc-f87bcda0c300', 'function': {'arguments': '{"genre":"Sports Drama","release_year":2019,"title":"Jersey"}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 128, 'prompt_tokens': 155, 'total_tokens': 283, 'completion_time': 0.129299535, 'completion_tokens_details': {'reasoning_tokens': 94}, 'prompt_time': 0.007510384, 'prompt_tokens_details': None, 'queue_time': 0.052121874, 'total_time': 0.136809919}, 'model_name': 'openai/gpt-oss-20b', 'syst

### Nested Structured Output

In [ ]:
class Player(BaseModel):
    name: str = Field(..., description="The name of the player")
    role: str = Field(..., description="The role of the player in the team. Ex: Batsman, Bowler, All-rounder, Wicket-keeper, Captain")

class Team(BaseModel):
    name: str = Field(..., description="The name of the team")
    players: list[Player] = Field(..., description="List of players in the team")   # List of players in the team

structured_response_nested = groq_model.with_structured_output(Team).invoke("Give me best playing 11 of the Indian cricket team in batting order.")
for player in structured_response_nested.players:
    print(f"{player.name} - {player.role}")

Rohit Sharma - Captain, Batsman
KL Rahul - Wicket-keeper, Batsman
Virat Kohli - Batsman
Hardik Pandya - All-rounder
Shreyas Iyer - Batsman
Yashasvi Jaiswal - Batsman
Rishabh Pant - All-rounder
Bhuvneshwar Kumar - Bowler
Rahul Chahar - Bowler
Jasprit Bumrah - Bowler
Jaydev Unadkat - Bowler
